In [1]:
using SpecialFunctions
using LinearAlgebra
using Plots, LaTeXStrings
using Struve
using KrylovKit
using SparseArrays
using QuadGK
using ForwardDiff
using FFTW
using Statistics
using Measures
using TensorOperations
include("module_graphene.jl")
using .exciton
using CSV, DataFrames

# Fig 2a

In [2]:
const lambda=0.75
const r0=1.5

delta_value = 1
t0 = 0.5 * delta_value
t1 = 0.9 * t0
t2 = 1 * t0
t3 = t0

alphalist = 0.0:0.1:1.0
eigenvalue_number = 80
energy_list = zeros(Float64, length(alphalist), eigenvalue_number)

Nsample = 24
lattice = GrapheneLattice();
VInt = InteractionMatrix(lattice, Nsample; lambda, r0)

s = 0
for alphacoefficient in alphalist
    graphene_sys = GrapheneBSE(lattice, [alphacoefficient / Nsample, 0.0], [t1, t2, t3], delta_value, Nsample)
    graphene_sys = add_bse_kernel(graphene_sys, VInt)
    s += 1
    println(alphacoefficient)

    bsemat = graphene_sys.BSEKernel
    xlen = size(bsemat)[1]
    x0 = rand(xlen)
    valslist, vecslist, info = eigsolve(bsemat, x0, eigenvalue_number, :SR,
        krylovdim=120, tol=1e-8, maxiter=20,
        verbosity=0, ishermitian=true)
    energy_list[s, :] .= real(valslist[1:eigenvalue_number])
end

0.0
0.1
0.2
0.3
0.4
0.5
0.6
0.7
0.8
0.9
1.0


In [3]:
energy_t = energy_list' 
df = DataFrame(Ordinal = 1:size(energy_t,1))

for (i, k) in enumerate(alphalist)
    df[!, string(k)] = energy_t[:, i]  # column name = kappa value
end

CSV.write("fig_2a.csv", df)

"fig_2a.csv"

# Fig 2ainset

In [ ]:
delocalized_list = [10, 18, 35] # above gap

3-element Vector{Int64}:
 10
 18
 35

In [5]:
alphalist = [0.0, 0.5]
eigenvalue_number = 80
roundval = 5
starting = 12
Nsample_list = collect(starting:6:(starting+6*roundval))
ET_list = zeros(Float64, length(Nsample_list), 5)

ind = 0
localized_list = [1, 3]
for (lsize, Nsample) in enumerate(Nsample_list)
    ind += 1
    println("This is round $ind")
    e_list = zeros(Float64, 2, 5)

    lattice = GrapheneLattice()
    VInt = InteractionMatrix(lattice, Nsample; lambda, r0)
    for (s, α) in enumerate(alphalist)
        graphene_sys = GrapheneBSE(lattice, [α / Nsample, 0], [t1, t2, t3], delta_value, Nsample)
        graphene_sys = add_bse_kernel(graphene_sys, VInt)
        bsemat = graphene_sys.BSEKernel
        xlen = size(bsemat)[1]
        x0 = rand(xlen)

        enum = 30
        if Nsample > 15
            enum = 80
        end

        valslist, vecslist, info = eigsolve(bsemat, x0, enum, :SR,
            krylovdim=140, tol=1e-8, maxiter=20,
            verbosity=0, ishermitian=true)

        dej = 0
        for items in localized_list
            dej += 1
            e_list[s, dej] = real(valslist[items])

        end

        offset = count(<(1), valslist[:])

        for items in delocalized_list
            dej += 1
            e_list[s, dej] = real(valslist[offset+items])
        end
    end

    # Save scaled energy difference
    ET_list[lsize, :] .= (e_list[1, :] .- e_list[2, :]) .* Nsample^2
end


This is round 1
This is round 2
This is round 3
This is round 4
This is round 5
This is round 6


In [7]:
df = DataFrame(
    size = Nsample_list,
    localized_1 = ET_list[:,1],
    localized_2 = ET_list[:,2],
    delocalized_1 = ET_list[:,3],
    delocalized_2 = ET_list[:,4],
    delocalized_3 = ET_list[:,5]
)

CSV.write("fig_2a_inset.csv", df)

"fig_2a_inset.csv"

# Fig 2b

In [ ]:
alphalist = 0.000:0.1:1.000
wilson_number = 80
Nsample = 24
eigenvalue_number = 80

wilson_list = zeros(Float64, length(alphalist), wilson_number)

# Reciprocal lattice vectors
lattice = GrapheneLattice()
VInt = InteractionMatrix(lattice, Nsample; lambda,r0)

offset=0
s = 0
for α in alphalist
    s += 1
    println(α)
    flush(stdout)
    graphene_sys = GrapheneBSE(lattice, [α / Nsample, 0], [t1, t2, t3], delta_value, Nsample)
    graphene_sys = add_bse_kernel(graphene_sys, VInt)
    bsemat = graphene_sys.BSEKernel
    xlen = size(bsemat)[1]
    x0 = rand(xlen)
    valslist, vecslist, info = eigsolve(bsemat, x0, eigenvalue_number, :SR,
        krylovdim=120, tol=1e-8, maxiter=20,
        verbosity=0, ishermitian=true)

    offset = count(<(1), valslist[:])

    precomputed = Dict{Tuple{Int,Int},Tuple{Matrix,Matrix,Matrix}}()

    for xdim = 1:Nsample, ydim = 1:Nsample
        b1, b2 = lattice.b1, lattice.b2
        R1, R2 = lattice.R1, lattice.R2
        k = (b1 * xdim + b2 * ydim) / Nsample
        kx, ky = k
        umat, _ = exciton.eigensystem(k; t1, t2, t3, delta=delta_value,
            A=[α / Nsample, 0], R1, R2, lattice)
        Avec = α * b1 / Nsample
        jx = jxop(kx, ky; t1, t2, t3, delta=delta_value, A=Avec, lattice)
        jy = jyop(kx, ky; t1, t2, t3, delta=delta_value, A=Avec, lattice)
        precomputed[(xdim, ydim)] = (umat, jx, jy)
    end

    for i = 1:wilson_number
        psi = vecslist[i]
        pump_sum = 0.0
        relax_sum = 0.0
        for xdim = 1:Nsample, ydim = 1:Nsample
            umat, jx, jy = precomputed[(xdim, ydim)]
            index = ham_index(xdim, ydim; xlength=Nsample, ylength=Nsample)

            pump_sum += (umat'*(jx+sqrt(3)*jy)*umat)[2, 1] * psi[index]
            relax_sum += (umat'*(jx-sqrt(3)*jy)*umat)[2, 1] * psi[index]
        end
        wilson_list[s, i] = angle(pump_sum * conj(relax_sum))
    end
end

0.0
0.1
0.2
0.3
0.4
0.5
0.6
0.7
0.8
0.9
1.0


30

In [10]:
df = DataFrame(
    kappa_list = alphalist,
    localized_1 = wrap_to_pi.(wilson_list[:,1]),
    localized_2 = wrap_to_positive.(wilson_list[:,3]),
    delocalized_1 = wrap_to_positive.(wilson_list[:,offset+delocalized_list[1]]),
    delocalized_2 = wrap_to_positive.(wilson_list[:,offset+delocalized_list[2]]),
    delocalized_3 = wrap_to_pi.(wilson_list[:,offset+delocalized_list[3]])
)

CSV.write("fig_2b.csv", df)

"fig_2b.csv"

# Fig 2c

In [70]:
alphalist = 0.01:0.1:1
selected_states = [1, 3]

ET_Wilson_tensor = zeros(Float64, length(Nsample_list), length(alphalist), length(selected_states)+3)

for (lsize, Nsample) in enumerate(Nsample_list)
    println("Running for Nsample = $Nsample")
    flush(stdout)
    lattice = GrapheneLattice()
    VInt = InteractionMatrix(lattice, Nsample; lambda, r0)

    for (jue, α) in enumerate(alphalist)

        graphene_sys = GrapheneBSE(lattice, [α / Nsample, 0], [t1, t2, t3], delta_value, Nsample)
        graphene_sys = add_bse_kernel(graphene_sys, VInt)
        bsemat = graphene_sys.BSEKernel
        xlen = size(bsemat)[1]
        x0 = rand(xlen)
        enum=0
        if Nsample < 15
            enum = 30
        else
            enum = 80
        end
        valslist, vecslist, info = eigsolve(bsemat, x0, enum, :SR,
            krylovdim=140, tol=1e-8, maxiter=20,
            verbosity=0, ishermitian=true)

        offset = count(<(1), valslist[:])

        precomputed = Dict{Tuple{Int,Int},Tuple{Matrix,Matrix,Matrix}}()

        for xdim = 1:Nsample, ydim = 1:Nsample
            b1, b2 = lattice.b1, lattice.b2
            R1, R2 = lattice.R1, lattice.R2
            k = (b1 * xdim + b2 * ydim) / Nsample
            kx, ky = k
            umat, _ = exciton.eigensystem(k; t1, t2, t3, delta=delta_value,
                A=[α / Nsample, 0], R1, R2, lattice)
            Avec = α * b1 / Nsample
            jx = jxop(kx, ky; t1, t2, t3, delta=delta_value, A=Avec, lattice)
            jy = jyop(kx, ky; t1, t2, t3, delta=delta_value, A=Avec, lattice)
            precomputed[(xdim, ydim)] = (umat, jx, jy)
        end

        sindicator = 0
        # Loop over selected states
        for (sidx, si) in enumerate(selected_states)
            sindicator += 1
            psi = vecslist[si]
            pump_sum = 0.0
            relax_sum = 0.0

            for xdim = 1:Nsample, ydim = 1:Nsample
                umat, jx, jy = precomputed[(xdim, ydim)]
                index = ham_index(xdim, ydim; xlength=Nsample, ylength=Nsample)

                pump_sum += (umat'*(jx+sqrt(3)*jy)*umat)[2, 1] * psi[index]
                relax_sum += (umat'*(jx-sqrt(3)*jy)*umat)[2, 1] * psi[index]
            end
            ET_Wilson_tensor[lsize, jue, sidx] = angle(pump_sum * conj(relax_sum))
        end

        for items in delocalized_list
            sindicator += 1

            psi = vecslist[offset+items]
            pump_sum = 0.0
            relax_sum = 0.0

            for xdim = 1:Nsample, ydim = 1:Nsample
                umat, jx, jy = precomputed[(xdim, ydim)]
                index = ham_index(xdim, ydim; xlength=Nsample, ylength=Nsample)

                pump_sum += (umat'*(jx+sqrt(3)*jy)*umat)[2, 1] * psi[index]
                relax_sum += (umat'*(jx-sqrt(3)*jy)*umat)[2, 1] * psi[index]
            end
            ET_Wilson_tensor[lsize, jue, sindicator] = angle(pump_sum * conj(relax_sum))
        end
    end
end

# Initialize container for standard deviations


Running for Nsample = 12
Running for Nsample = 18
Running for Nsample = 24
Running for Nsample = 30
Running for Nsample = 36
Running for Nsample = 42


In [72]:
std_over_alpha = zeros(Float64, length(Nsample_list), length(selected_states)+3)
# Loop to compute std dev over α for each L and each eigenstate
sset = [2]
for (lsize, _) in enumerate(Nsample_list)
    for localized_ind = 1:(length(selected_states)+3)
        num = ET_Wilson_tensor[lsize, 1, localized_ind]
        data = 0
        if abs(num) < 3
            data = wrap_to_pi.(ET_Wilson_tensor[lsize, :, localized_ind])
        elseif abs(abs(num) - pi) < 3
            data = wrap_to_positive.(ET_Wilson_tensor[lsize, :, localized_ind])
        end
        std_over_alpha[lsize, localized_ind] = std(data)
    end
end

In [75]:
Nsample_list

6-element Vector{Int64}:
 12
 18
 24
 30
 36
 42

In [76]:
df = DataFrame(
    size_list = Nsample_list,
    localized_1 = std_over_alpha[:,1],
    localized_2 = std_over_alpha[:,2],
    delocalized_1 = std_over_alpha[:,3],
    delocalized_2 = std_over_alpha[:,4],
    delocalized_3 = std_over_alpha[:,5]
)

CSV.write("fig_2c.csv", df)

"fig_2c.csv"